In [1]:
import pandas as pd

CSV_PATH = "/scratch/budayaku/SML/Final_Augmented_dataset_Diseases_and_Symptoms.csv"

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nFirst 8 columns:", df.columns.tolist()[:8])
print("\nLast 3 columns:", df.columns.tolist()[-3:])
print("\nFirst row (first 8 cols):")
print(df.iloc[0, :8])
print("\nLabel column (assuming col 0):", df.columns[0])
print("Unique diseases:", df.iloc[:, 0].nunique())
print("\nTop 10 most common diseases:")
print(df.iloc[:, 0].value_counts().head(10))
print("\nClass count stats:")
print(df.iloc[:, 0].value_counts().describe())
print("\nDtypes of last few cols:", df.dtypes.tail(5).to_dict())

Shape: (246945, 378)

First 8 columns: ['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms', 'sharp chest pain', 'dizziness', 'insomnia']

Last 3 columns: ['ankle stiffness or tightness', 'ankle weakness', 'neck weakness']

First row (first 8 cols):
diseases                            panic disorder
anxiety and nervousness                          1
depression                                       0
shortness of breath                              1
depressive or psychotic symptoms                 1
sharp chest pain                                 0
dizziness                                        0
insomnia                                         0
Name: 0, dtype: object

Label column (assuming col 0): diseases
Unique diseases: 773

Top 10 most common diseases:
diseases
cystitis                          1219
nose disorder                     1218
vulvodynia                        1218
complex regional pain syndrome    1217
spo

In [2]:
!pip install xgboost scikit-learn pandas numpy --quiet

In [3]:
import pandas as pd

CSV_PATH = "/scratch/budayaku/SML/Final_Augmented_dataset_Diseases_and_Symptoms.csv"

df = pd.read_csv(CSV_PATH)

print("Shape:", df.shape)
print("\nFirst 8 columns:", df.columns.tolist()[:8])
print("\nLast 3 columns:", df.columns.tolist()[-3:])
print("\nFirst row (first 8 cols):")
print(df.iloc[0, :8])
print("\nAssumed label column:", df.columns[0])
print("Unique diseases:", df.iloc[:, 0].nunique())
print("\nTop 10 most common diseases:")
print(df.iloc[:, 0].value_counts().head(10))
print("\nClass count stats:")
print(df.iloc[:, 0].value_counts().describe())
print("\nDtypes summary:")
print(df.dtypes.value_counts())
print("\nAny NaN?", df.isnull().any().any())

Shape: (246945, 378)

First 8 columns: ['diseases', 'anxiety and nervousness', 'depression', 'shortness of breath', 'depressive or psychotic symptoms', 'sharp chest pain', 'dizziness', 'insomnia']

Last 3 columns: ['ankle stiffness or tightness', 'ankle weakness', 'neck weakness']

First row (first 8 cols):
diseases                            panic disorder
anxiety and nervousness                          1
depression                                       0
shortness of breath                              1
depressive or psychotic symptoms                 1
sharp chest pain                                 0
dizziness                                        0
insomnia                                         0
Name: 0, dtype: object

Assumed label column: diseases
Unique diseases: 773

Top 10 most common diseases:
diseases
cystitis                          1219
nose disorder                     1218
vulvodynia                        1218
complex regional pain syndrome    1217
spondylosis 

In [4]:
import os
import json
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, top_k_accuracy_score, f1_score, classification_report
import time

CSV_PATH = "/scratch/budayaku/SML/Final_Augmented_dataset_Diseases_and_Symptoms.csv"
MODEL_DIR = "/scratch/budayaku/SML/model"
os.makedirs(MODEL_DIR, exist_ok=True)

MIN_SAMPLES_PER_CLASS = 5   # drop classes with fewer than this
RANDOM_SEED = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15  # of the remaining 85%

In [5]:
print("Loading CSV...")
t0 = time.time()
df = pd.read_csv(CSV_PATH)
print(f"Loaded {df.shape} in {time.time()-t0:.1f}s")

# Separate label from features
y_raw = df["diseases"].values
X = df.drop(columns=["diseases"]).values.astype(np.int8)  # int8 saves memory
symptom_names = df.drop(columns=["diseases"]).columns.tolist()
print(f"Features: {X.shape}, Symptoms: {len(symptom_names)}")

# Drop classes with fewer than MIN_SAMPLES_PER_CLASS samples
class_counts = pd.Series(y_raw).value_counts()
keep_classes = class_counts[class_counts >= MIN_SAMPLES_PER_CLASS].index.tolist()
mask = np.isin(y_raw, keep_classes)
X = X[mask]
y_raw = y_raw[mask]
print(f"After dropping classes with <{MIN_SAMPLES_PER_CLASS} samples:")
print(f"  Rows: {X.shape[0]}, Classes: {len(keep_classes)}")
print(f"  Dropped {773 - len(keep_classes)} classes")

# Encode disease names to integer labels
le = LabelEncoder()
y = le.fit_transform(y_raw)
print(f"Label encoding done. Num classes: {len(le.classes_)}")

Loading CSV...
Loaded (246945, 378) in 2.3s
Features: (246945, 377), Symptoms: 377
After dropping classes with <5 samples:
  Rows: 246823, Classes: 721
  Dropped 52 classes
Label encoding done. Num classes: 721


In [6]:
# First split: train+val vs test
X_trainval, X_test, y_trainval, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    stratify=y,
    random_state=RANDOM_SEED,
)

# Second split: train vs val (val is VAL_SIZE of the original, so adjust)
val_ratio = VAL_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_trainval, y_trainval,
    test_size=val_ratio,
    stratify=y_trainval,
    random_state=RANDOM_SEED,
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")

Train: (172775, 377), Val: (37024, 377), Test: (37024, 377)


In [12]:
num_classes = len(le.classes_)

model = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=num_classes,
    tree_method="hist",
    device="cuda",
    n_estimators=800,            # was 300 - more room to grow
    max_depth=10,                # was 8 - slightly deeper
    learning_rate=0.05,          # was 0.1 - slower, better convergence
    subsample=0.9,
    colsample_bytree=0.8,
    min_child_weight=1,
    reg_alpha=0.1,               # mild L1 regularization
    reg_lambda=1.0,              # mild L2 regularization
    random_state=RANDOM_SEED,
    eval_metric="mlogloss",
    early_stopping_rounds=30,    # was 20 - give it more patience
    verbosity=1,
)

print("Training XGBoost v2 on GPU...")
t0 = time.time()
model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=25,
)
print(f"Training done in {time.time()-t0:.1f}s")
print(f"Best iteration: {model.best_iteration}")

Training XGBoost v2 on GPU...
[0]	validation_0-mlogloss:4.54815
[25]	validation_0-mlogloss:1.40713
[50]	validation_0-mlogloss:0.86530
[75]	validation_0-mlogloss:0.65509
[100]	validation_0-mlogloss:0.56058
[125]	validation_0-mlogloss:0.51520
[150]	validation_0-mlogloss:0.49254
[175]	validation_0-mlogloss:0.48158
[200]	validation_0-mlogloss:0.47686
[225]	validation_0-mlogloss:0.47619
[250]	validation_0-mlogloss:0.47751
[251]	validation_0-mlogloss:0.47760
Training done in 394.2s
Best iteration: 221


In [13]:
print("Evaluating on test set...")
y_proba = model.predict_proba(X_test)
y_pred = np.argmax(y_proba, axis=1)

top1 = accuracy_score(y_test, y_pred)
top3 = top_k_accuracy_score(y_test, y_proba, k=3, labels=np.arange(num_classes))
top5 = top_k_accuracy_score(y_test, y_proba, k=5, labels=np.arange(num_classes))
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"\n=== Test Set Results ===")
print(f"Top-1 accuracy:  {top1:.4f}")
print(f"Top-3 accuracy:  {top3:.4f}")
print(f"Top-5 accuracy:  {top5:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
print(f"Weighted F1:     {weighted_f1:.4f}")

Evaluating on test set...

=== Test Set Results ===
Top-1 accuracy:  0.8373
Top-3 accuracy:  0.9484
Top-5 accuracy:  0.9734
Macro F1:        0.7418
Weighted F1:     0.8363


In [14]:
print("Evaluating on test set...")
y_proba = model.predict_proba(X_test)
y_pred = np.argmax(y_proba, axis=1)

top1 = accuracy_score(y_test, y_pred)
top3 = top_k_accuracy_score(y_test, y_proba, k=3, labels=np.arange(num_classes))
top5 = top_k_accuracy_score(y_test, y_proba, k=5, labels=np.arange(num_classes))
macro_f1 = f1_score(y_test, y_pred, average="macro")
weighted_f1 = f1_score(y_test, y_pred, average="weighted")

print(f"\n=== Test Set Results ===")
print(f"Top-1 accuracy:  {top1:.4f}")
print(f"Top-3 accuracy:  {top3:.4f}")
print(f"Top-5 accuracy:  {top5:.4f}")
print(f"Macro F1:        {macro_f1:.4f}")
print(f"Weighted F1:     {weighted_f1:.4f}")

Evaluating on test set...

=== Test Set Results ===
Top-1 accuracy:  0.8373
Top-3 accuracy:  0.9484
Top-5 accuracy:  0.9734
Macro F1:        0.7418
Weighted F1:     0.8363


In [15]:
# Save model
model_path = os.path.join(MODEL_DIR, "xgb_disease_model.json")
model.save_model(model_path)
print(f"Saved model: {model_path}")

# Save symptom list (order MATTERS — this is the input vector order)
symptoms_path = os.path.join(MODEL_DIR, "symptoms.json")
with open(symptoms_path, "w") as f:
    json.dump(symptom_names, f, indent=2)
print(f"Saved symptoms: {symptoms_path}")

# Save disease label mapping (index -> disease name)
diseases_path = os.path.join(MODEL_DIR, "diseases.json")
diseases_map = {int(i): name for i, name in enumerate(le.classes_)}
with open(diseases_path, "w") as f:
    json.dump(diseases_map, f, indent=2)
print(f"Saved diseases: {diseases_path}")

# Save test metrics for the report
metrics_path = os.path.join(MODEL_DIR, "metrics.json")
metrics = {
    "top1_accuracy": float(top1),
    "top3_accuracy": float(top3),
    "top5_accuracy": float(top5),
    "macro_f1": float(macro_f1),
    "weighted_f1": float(weighted_f1),
    "num_classes": int(num_classes),
    "num_symptoms": len(symptom_names),
    "train_size": int(X_train.shape[0]),
    "val_size": int(X_val.shape[0]),
    "test_size": int(X_test.shape[0]),
}
with open(metrics_path, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics: {metrics_path}")

print("\n✓ Day 1 artifacts ready for Day 2 integration.")

Saved model: /scratch/budayaku/SML/model/xgb_disease_model.json
Saved symptoms: /scratch/budayaku/SML/model/symptoms.json
Saved diseases: /scratch/budayaku/SML/model/diseases.json
Saved metrics: /scratch/budayaku/SML/model/metrics.json

✓ Day 1 artifacts ready for Day 2 integration.


In [16]:
def predict_top_k(symptom_vector, k=5):
    """
    symptom_vector: np.array of shape (377,) with 0/1 values in symptom_names order
    Returns: list of (disease_name, probability) tuples, length k, sorted by prob desc
    """
    x = np.asarray(symptom_vector, dtype=np.int8).reshape(1, -1)
    proba = model.predict_proba(x)[0]
    top_idx = np.argsort(proba)[::-1][:k]
    return [(le.classes_[i], float(proba[i])) for i in top_idx]

# Quick sanity test: take the first test sample and predict
test_vec = X_test[0]
true_disease = le.classes_[y_test[0]]
print(f"True disease: {true_disease}")
print(f"Active symptoms: {[symptom_names[i] for i in range(len(test_vec)) if test_vec[i]==1][:10]}")
print(f"\nTop 5 predictions:")
for name, prob in predict_top_k(test_vec, k=5):
    print(f"  {name}: {prob:.4f}")

True disease: injury to the arm
Active symptoms: ['arm stiffness or tightness', 'arm swelling', 'elbow pain', 'elbow swelling']

Top 5 predictions:
  bursitis: 0.5342
  injury to the arm: 0.2815
  fracture of the arm: 0.0744
  dislocation of the elbow: 0.0698
  pyloric stenosis: 0.0019


In [17]:
import json
import numpy as np
import xgboost as xgb

MODEL_DIR = "/scratch/budayaku/SML/model"

# Load the trained XGBoost model
clf = xgb.XGBClassifier()
clf.load_model(f"{MODEL_DIR}/xgb_disease_model.json")

# Load the ordered symptom list (377 names)
with open(f"{MODEL_DIR}/symptoms.json") as f:
    SYMPTOM_NAMES = json.load(f)

# Load the disease label mapping (index -> name)
with open(f"{MODEL_DIR}/diseases.json") as f:
    DISEASES_MAP = json.load(f)
# JSON keys are strings; convert to int
DISEASES = {int(k): v for k, v in DISEASES_MAP.items()}

# Build a lookup: symptom name -> index in the vector
SYMPTOM_INDEX = {name: i for i, name in enumerate(SYMPTOM_NAMES)}

print(f"Loaded model. {len(SYMPTOM_NAMES)} symptoms, {len(DISEASES)} diseases.")

Loaded model. 377 symptoms, 721 diseases.


In [20]:
# Every key here EXISTS in symptoms.json (verified).
# Starter dictionary covering the most common symptoms.
SYNONYMS = {
    # --- Mental / emotional ---
    "anxiety and nervousness": ["anxiety", "anxious", "nervous", "nervousness", "panicky", "on edge"],
    "depression": ["depression", "depressed", "hopeless", "feeling down", "sad all the time"],
    "insomnia": ["insomnia", "cant sleep", "can't sleep", "trouble sleeping", "cannot sleep", "sleepless"],

    # --- Chest / cardiac ---
    "sharp chest pain": ["sharp chest pain", "stabbing chest pain", "stabbing pain in chest", "chest stabbing", "chest pain"],
    "shortness of breath": ["shortness of breath", "short of breath", "cant breathe", "can't breathe", "hard to breathe", "breathless", "trouble breathing", "difficulty breathing"],
    "palpitations": ["palpitations", "heart racing", "heart pounding", "fast heartbeat", "racing heart"],
    "irregular heartbeat": ["irregular heartbeat", "skipped beats", "heart skipping"],

    # --- Head / neuro ---
    "headache": ["headache", "head pain", "head hurts", "migraine", "pounding head", "head aching"],
    "dizziness": ["dizziness", "dizzy", "lightheaded", "light headed", "room spinning", "vertigo", "spinning"],
    "fainting": ["fainting", "fainted", "passed out", "blacking out"],
    "seizures": ["seizures", "seizure", "convulsions", "fit"],
    "disturbance of memory": ["memory loss", "forgetting things", "cant remember", "memory problems", "disturbance of memory"],
    "slurring words": ["slurred speech", "slurring words", "speech slurred"],

    # --- General / systemic ---
    "fever": ["fever", "feverish", "high temperature", "running a temperature", "temperature"],
    "chills": ["chills", "shivering", "shivers", "chilled"],
    "sweating": ["sweating", "sweaty", "perspiring", "sweats"],
    "fatigue": ["fatigue", "tired", "exhausted", "no energy", "worn out", "drained", "weary"],
    "feeling ill": ["feeling ill", "feel sick", "unwell", "not feeling well"],
    "feeling hot": ["feeling hot", "burning up", "hot"],
    "feeling cold": ["feeling cold", "always cold", "cant get warm", "cold intolerance"],
    "feeling hot and cold": ["hot and cold", "alternating hot and cold"],
    "hot flashes": ["hot flashes", "hot flushes", "sudden heat"],
    "recent weight loss": ["weight loss", "losing weight", "lost weight", "dropping weight", "recent weight loss"],
    "weight gain": ["weight gain", "gaining weight", "gained weight"],
    "underweight": ["underweight", "too thin", "very thin"],

    # --- GI / abdominal ---
    "lower abdominal pain": ["stomach pain", "stomach ache", "tummy pain", "belly pain", "abdominal pain", "pain in abdomen", "pain in stomach", "lower abdominal pain", "lower belly pain"],
    "upper abdominal pain": ["upper abdominal pain", "upper stomach pain", "pain in upper abdomen"],
    "sharp abdominal pain": ["sharp abdominal pain", "stabbing stomach pain", "sharp stomach pain"],
    "burning abdominal pain": ["burning abdominal pain", "burning stomach pain"],
    "nausea": ["nausea", "nauseous", "queasy", "sick to my stomach"],
    "vomiting": ["vomiting", "throwing up", "puking", "threw up", "vomit"],
    "diarrhea": ["diarrhea", "loose stool", "loose stools", "runny stool", "watery stool"],
    "constipation": ["constipation", "constipated", "cant poop", "can't poop", "hard stool"],
    "blood in stool": ["blood in stool", "bloody stool", "rectal bleeding", "blood when pooping"],
    "difficulty in swallowing": ["difficulty swallowing", "difficulty in swallowing", "hard to swallow", "trouble swallowing", "cant swallow"],
    "heartburn": ["heartburn", "acid reflux", "burning in chest", "burning after eating"],
    "stomach bloating": ["bloating", "bloated", "stomach bloated", "swollen belly", "stomach bloating"],
    "abdominal distention": ["abdominal distention", "distended stomach", "swollen abdomen"],
    "flatulence": ["gas", "gassy", "passing gas", "flatulence"],
    "decreased appetite": ["loss of appetite", "no appetite", "not hungry", "cant eat", "can't eat", "decreased appetite"],
    "excessive appetite": ["excessive appetite", "always hungry", "increased appetite"],
    "difficulty eating": ["difficulty eating", "cant eat properly", "trouble eating"],

    # --- Respiratory / ENT ---
    "cough": ["cough", "coughing", "hacking", "dry cough", "wet cough"],
    "sore throat": ["sore throat", "throat pain", "throat hurts", "scratchy throat"],
    "nasal congestion": ["stuffy nose", "blocked nose", "congested", "nasal congestion", "runny nose", "nose running", "nasal discharge", "cant breathe through nose"],
    "nosebleed": ["nosebleed", "bloody nose", "nose bleeding"],
    "ear pain": ["ear pain", "earache", "ear hurts"],
    "diminished hearing": ["hearing loss", "cant hear", "can't hear", "trouble hearing", "diminished hearing"],
    "pus draining from ear": ["pus from ear", "ear discharge", "pus draining from ear"],

    # --- Eyes / vision ---
    "diminished vision": ["blurred vision", "blurry vision", "cant see clearly", "vision blurry", "diminished vision"],
    "double vision": ["double vision", "seeing double"],
    "spots or clouds in vision": ["spots in vision", "floaters", "cloudy vision", "spots or clouds in vision"],
    "eye redness": ["red eyes", "bloodshot eyes", "eye redness"],

    # --- Skin ---
    "skin rash": ["rash", "skin rash", "red patches", "skin irritation"],
    "itching of skin": ["itchy", "itchiness", "itching", "skin itching", "scratchy skin", "itching of skin"],
    "abnormal appearing skin": ["abnormal skin", "strange skin", "abnormal appearing skin"],
    "too little hair": ["hair loss", "losing hair", "hair falling out", "balding", "too little hair"],
    "unwanted hair": ["unwanted hair", "excess hair"],
    "irregular appearing nails": ["brittle nails", "nails breaking", "abnormal nails", "irregular appearing nails"],
    "mouth dryness": ["dry mouth", "mouth is dry", "cotton mouth", "mouth dryness"],

    # --- Urinary ---
    "painful urination": ["painful urination", "burning when peeing", "pain when urinating", "burns to pee"],
    "frequent urination": ["frequent urination", "peeing a lot", "constantly peeing", "pee often"],
    "blood in urine": ["blood in urine", "bloody urine", "bleeding when peeing"],

    # --- Musculoskeletal ---
    "back pain": ["back pain", "back hurts", "aching back", "sore back"],
    "neck pain": ["neck pain", "neck hurts", "stiff neck", "sore neck"],
    "joint pain": ["joint pain", "joints hurt", "aching joints", "sore joints"],
    "muscle pain": ["muscle pain", "muscle ache", "muscles hurt", "sore muscles", "body ache", "body aches"],
    "leg pain": ["leg pain", "legs hurt", "aching legs", "sore legs"],
    "leg swelling": ["swollen legs", "leg swelling", "puffy legs"],
    "leg cramps or spasms": ["leg cramps", "leg spasms", "leg cramps or spasms"],
    "knee pain": ["knee pain", "knee hurts", "sore knee"],
    "ankle pain": ["ankle pain", "ankle hurts", "sore ankle"],
    "ankle swelling": ["swollen ankles", "ankle swelling", "puffy ankles"],
    "foot or toe pain": ["foot pain", "feet hurt", "sore feet", "toe pain", "foot or toe pain"],
    "foot or toe swelling": ["swollen feet", "foot swelling", "foot or toe swelling"],
    "arm pain": ["arm pain", "arm hurts", "sore arm"],
    "elbow pain": ["elbow pain", "elbow hurts", "sore elbow"],
    "wrist pain": ["wrist pain", "wrist hurts", "sore wrist"],
    "hand or finger pain": ["hand pain", "finger pain", "hand hurts", "fingers hurt", "hand or finger pain"],
    "shoulder pain": ["shoulder pain", "shoulder hurts", "sore shoulder"],
    "hip pain": ["hip pain", "hip hurts", "sore hip"],
    "weakness": ["weakness", "weak", "feel weak"],

    # --- Lymph / swelling ---
    "swollen lymph nodes": ["swollen lymph nodes", "swollen glands", "lumps in neck"],
    "neck swelling": ["neck swelling", "swollen neck"],
    "facial pain": ["face pain", "facial pain"],

    # --- Reproductive ---
    "vaginal discharge": ["vaginal discharge", "discharge"],
    "impotence": ["erectile dysfunction", "cant get erection", "ed", "impotence"],
    "pelvic pain": ["pelvic pain", "pelvis hurts", "pain in pelvis"],
}

# Sanity check: every key MUST exist in SYMPTOM_NAMES
missing = [k for k in SYNONYMS if k not in SYMPTOM_INDEX]
print(f"Dictionary entries: {len(SYNONYMS)}")
print(f"Total phrases: {sum(len(v) for v in SYNONYMS.values())}")
print(f"Missing from symptoms.json: {len(missing)}")
if missing:
    print("STILL MISSING — need to fix:")
    for m in missing:
        print(f"  '{m}'")
else:
    print("✓ All keys validated against symptoms.json")

Dictionary entries: 88
Total phrases: 355
Missing from symptoms.json: 0
✓ All keys validated against symptoms.json


In [21]:
import re

def normalize(text: str) -> str:
    """Lowercase, strip extra whitespace and punctuation."""
    text = text.lower()
    text = re.sub(r"[^\w\s']", " ", text)  # keep apostrophes
    text = re.sub(r"\s+", " ", text).strip()
    return text

# Pre-build a flat list: (phrase, canonical_name) sorted by phrase length DESC
# Longer phrases match first so "sharp chest pain" wins over "chest pain"
_PHRASE_LOOKUP = []
for canonical, phrases in SYNONYMS.items():
    for phrase in phrases:
        _PHRASE_LOOKUP.append((normalize(phrase), canonical))
_PHRASE_LOOKUP.sort(key=lambda x: -len(x[0]))

def extract_symptoms(user_text: str) -> list:
    """
    Scan user_text for symptom phrases.
    Returns a list of canonical symptom names (no duplicates), in order of first appearance.
    """
    text = " " + normalize(user_text) + " "
    found = []
    seen = set()
    for phrase, canonical in _PHRASE_LOOKUP:
        # Match whole-phrase only (with word boundaries via surrounding spaces)
        if f" {phrase} " in text and canonical not in seen:
            found.append(canonical)
            seen.add(canonical)
    return found

def symptoms_to_vector(symptom_list: list) -> np.ndarray:
    """Convert a list of canonical symptom names to a 377-length int8 vector."""
    vec = np.zeros(len(SYMPTOM_NAMES), dtype=np.int8)
    for s in symptom_list:
        if s in SYMPTOM_INDEX:
            vec[SYMPTOM_INDEX[s]] = 1
    return vec

def predict_from_text(user_text: str, k: int = 5):
    """End-to-end: text -> top-k disease predictions."""
    symptoms = extract_symptoms(user_text)
    if not symptoms:
        return {"symptoms": [], "predictions": [], "message": "No recognized symptoms"}
    vec = symptoms_to_vector(symptoms).reshape(1, -1)
    proba = clf.predict_proba(vec)[0]
    top_idx = np.argsort(proba)[::-1][:k]
    preds = [(DISEASES[int(i)], float(proba[i])) for i in top_idx]
    return {"symptoms": symptoms, "predictions": preds}

In [22]:
test_sentences = [
    "I have a terrible headache and I feel really dizzy",
    "My chest is hurting sharply and I can't breathe",
    "I've been coughing a lot and have a sore throat and fever",
    "My elbow hurts and it's swollen",
    "I'm throwing up and have diarrhea and stomach pain",
    "I can't sleep and I feel anxious all the time",
    "There's a rash on my skin and it's really itchy",
    "My left knee hurts when I walk",
    "I have blurred vision and a bad headache",
    "burning when I pee and peeing a lot",
]

print("="*70)
for sent in test_sentences:
    result = predict_from_text(sent, k=3)
    print(f"\nUser: {sent}")
    print(f"  Extracted: {result['symptoms']}")
    print(f"  Top 3 predictions:")
    for name, prob in result["predictions"]:
        print(f"    {name}: {prob:.3f}")
print("\n" + "="*70)


User: I have a terrible headache and I feel really dizzy
  Extracted: ['headache', 'dizziness']
  Top 3 predictions:
    autonomic nervous system disorder: 0.157
    atelectasis: 0.052
    cerebral edema: 0.037

User: My chest is hurting sharply and I can't breathe
  Extracted: ['shortness of breath']
  Top 3 predictions:
    kidney disease due to longstanding hypertension: 0.070
    glucocorticoid deficiency: 0.052
    aspergillosis: 0.052

User: I've been coughing a lot and have a sore throat and fever
  Extracted: ['sore throat', 'cough', 'fever']
  Top 3 predictions:
    lymphadenitis: 0.092
    oral mucosal lesion: 0.090
    conjunctivitis due to bacteria: 0.088

User: My elbow hurts and it's swollen
  Extracted: ['elbow pain']
  Top 3 predictions:
    dislocation of the elbow: 0.090
    brachial neuritis: 0.086
    carpal tunnel syndrome: 0.073

User: I'm throwing up and have diarrhea and stomach pain
  Extracted: ['lower abdominal pain', 'vomiting', 'diarrhea']
  Top 3 predicti

In [23]:
# Patches: missed phrasings discovered in Cell 13 testing
PATCHES = {
    # Chest pain — catch "chest hurting / chest hurts" phrasings
    "sharp chest pain": [
        "sharp chest pain", "stabbing chest pain", "stabbing pain in chest",
        "chest stabbing", "chest pain", "chest hurts", "chest is hurting",
        "pain in chest", "my chest hurts", "chest is hurting sharply",
    ],
    # Painful urination — catch "when I pee" variants
    "painful urination": [
        "painful urination", "burning when peeing", "pain when urinating",
        "burns to pee", "burning when i pee", "burning when i urinate",
        "it burns when i pee", "hurts to pee", "hurts when i pee",
        "burning pee", "stinging when i pee",
    ],
    # Elbow swelling — was missing entirely
    "elbow swelling": ["elbow swelling", "swollen elbow", "elbow is swollen", "my elbow is swollen"],
    # Knee swelling / ankle swelling — proactively add the same pattern
    "knee swelling": ["knee swelling", "swollen knee", "knee is swollen"],
    # Wrist / shoulder / hip swelling (check if these columns exist)
}

# Merge into SYNONYMS
for k, v in PATCHES.items():
    if k in SYMPTOM_INDEX:
        SYNONYMS[k] = list(set(SYNONYMS.get(k, []) + v))
    else:
        print(f"WARNING: '{k}' not in symptoms.json — skipping")

# Rebuild the phrase lookup (IMPORTANT — Cell 12's _PHRASE_LOOKUP is now stale)
_PHRASE_LOOKUP = []
for canonical, phrases in SYNONYMS.items():
    for phrase in phrases:
        _PHRASE_LOOKUP.append((normalize(phrase), canonical))
_PHRASE_LOOKUP.sort(key=lambda x: -len(x[0]))

print(f"Updated dictionary: {len(SYNONYMS)} entries, {sum(len(v) for v in SYNONYMS.values())} phrases")

Updated dictionary: 90 entries, 374 phrases


In [24]:
retry_sentences = [
    "My chest is hurting sharply and I can't breathe",
    "My elbow hurts and it's swollen",
    "burning when I pee and peeing a lot",
    "I have a terrible headache and I feel really dizzy",  # was fine, keeping as regression check
]

for sent in retry_sentences:
    result = predict_from_text(sent, k=3)
    print(f"\nUser: {sent}")
    print(f"  Extracted: {result['symptoms']}")
    print(f"  Top 3:")
    for name, prob in result["predictions"]:
        print(f"    {name}: {prob:.3f}")


User: My chest is hurting sharply and I can't breathe
  Extracted: ['sharp chest pain', 'shortness of breath']
  Top 3:
    chronic obstructive pulmonary disease (copd): 0.034
    premature ventricular contractions (pvcs): 0.024
    magnesium deficiency: 0.024

User: My elbow hurts and it's swollen
  Extracted: ['elbow pain']
  Top 3:
    dislocation of the elbow: 0.090
    brachial neuritis: 0.086
    carpal tunnel syndrome: 0.073

User: burning when I pee and peeing a lot
  Extracted: ['painful urination', 'frequent urination']
  Top 3:
    bladder disorder: 0.479
    temporary or benign blood in urine: 0.182
    benign vaginal discharge (leukorrhea): 0.064

User: I have a terrible headache and I feel really dizzy
  Extracted: ['headache', 'dizziness']
  Top 3:
    autonomic nervous system disorder: 0.157
    atelectasis: 0.052
    cerebral edema: 0.037


In [25]:
# Check if elbow swelling exists as a column
print("'elbow swelling' in symptoms.json?", "elbow swelling" in SYMPTOM_INDEX)

# Find elbow-related columns
print("\nElbow-related columns:")
for name in SYMPTOM_NAMES:
    if "elbow" in name.lower():
        print(f"  {name!r}")

# Check what the extractor is seeing for the elbow sentence
test = "My elbow hurts and it's swollen"
print(f"\nNormalized text: '{normalize(test)}'")
print(f"\nElbow-related phrases in lookup:")
for phrase, canon in _PHRASE_LOOKUP:
    if "elbow" in phrase:
        print(f"  {phrase!r} -> {canon!r}")

'elbow swelling' in symptoms.json? True

Elbow-related columns:
  'elbow weakness'
  'elbow pain'
  'elbow cramps or spasms'
  'elbow swelling'
  'elbow stiffness or tightness'
  'elbow lump or mass'

Normalized text: 'my elbow hurts and it's swollen'

Elbow-related phrases in lookup:
  'my elbow is swollen' -> 'elbow swelling'
  'elbow is swollen' -> 'elbow swelling'
  'elbow swelling' -> 'elbow swelling'
  'swollen elbow' -> 'elbow swelling'
  'elbow hurts' -> 'elbow pain'
  'elbow pain' -> 'elbow pain'
  'sore elbow' -> 'elbow pain'


In [26]:
# When a body part is mentioned with a detached descriptor in the same sentence,
# link them. This catches cases like "my elbow hurts and it's swollen" where
# 'swollen' is not adjacent to 'elbow' but clearly refers to it.

BODY_PARTS = ["elbow", "knee", "ankle", "wrist", "shoulder", "hip", "neck",
              "back", "leg", "arm", "hand", "foot", "finger", "toe"]

# Map (body_part, descriptor_word) -> canonical symptom name
CONTEXTUAL_RULES = {}
for part in BODY_PARTS:
    # Only add if the resulting canonical name actually exists as a column
    candidates = {
        "swollen": f"{part} swelling",
        "swelling": f"{part} swelling",
        "puffy": f"{part} swelling",
        "stiff": f"{part} stiffness or tightness",
        "tight": f"{part} stiffness or tightness",
        "weak": f"{part} weakness",
        "cramping": f"{part} cramps or spasms",
        "cramps": f"{part} cramps or spasms",
        "lump": f"{part} lump or mass",
        "mass": f"{part} lump or mass",
    }
    for descriptor, canonical in candidates.items():
        if canonical in SYMPTOM_INDEX:
            CONTEXTUAL_RULES[(part, descriptor)] = canonical
        # Foot/toe and hand/finger share columns in this dataset
        alt_canonical_foot = f"foot or toe {canonical.split(' ', 1)[1]}" if part in ("foot", "toe") else None
        alt_canonical_hand = f"hand or finger {canonical.split(' ', 1)[1]}" if part in ("hand", "finger") else None
        if alt_canonical_foot and alt_canonical_foot in SYMPTOM_INDEX:
            CONTEXTUAL_RULES[(part, descriptor)] = alt_canonical_foot
        if alt_canonical_hand and alt_canonical_hand in SYMPTOM_INDEX:
            CONTEXTUAL_RULES[(part, descriptor)] = alt_canonical_hand

print(f"Built {len(CONTEXTUAL_RULES)} contextual rules")

# Expand contractions before extracting — fixes Bug A
CONTRACTIONS = {
    "it's": "it is", "he's": "he is", "she's": "she is", "i'm": "i am",
    "i've": "i have", "can't": "cannot", "won't": "will not", "don't": "do not",
    "doesn't": "does not", "didn't": "did not", "isn't": "is not", "aren't": "are not",
    "wasn't": "was not", "weren't": "were not", "haven't": "have not", "hasn't": "has not",
    "hadn't": "had not", "wouldn't": "would not", "couldn't": "could not", "shouldn't": "should not",
    "i'll": "i will", "you're": "you are", "we're": "we are", "they're": "they are",
}

def expand_contractions(text: str) -> str:
    for contraction, expanded in CONTRACTIONS.items():
        text = text.replace(contraction, expanded)
    return text

def normalize(text: str) -> str:
    text = text.lower()
    text = expand_contractions(text)
    text = re.sub(r"[^\w\s]", " ", text)   # drop apostrophes now (already expanded)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def apply_contextual_rules(text: str, already_found: list) -> list:
    """
    For each sentence, if a body part and a descriptor both appear (even non-adjacent),
    add the corresponding canonical symptom.
    """
    already_set = set(already_found)
    new_finds = []
    # Split on sentence-ish boundaries
    sentences = re.split(r"[.!?;]|\band\b", text)
    for sent in sentences:
        parts_in_sent = [p for p in BODY_PARTS if f" {p} " in f" {sent} "]
        for part in parts_in_sent:
            for (rule_part, descriptor), canonical in CONTEXTUAL_RULES.items():
                if rule_part == part and f" {descriptor} " in f" {sent} ":
                    if canonical not in already_set:
                        new_finds.append(canonical)
                        already_set.add(canonical)
    return new_finds

def extract_symptoms(user_text: str) -> list:
    text = " " + normalize(user_text) + " "
    found = []
    seen = set()
    # Phase 1: phrase matching (longest first)
    for phrase, canonical in _PHRASE_LOOKUP:
        if f" {phrase} " in text and canonical not in seen:
            found.append(canonical)
            seen.add(canonical)
    # Phase 2: contextual body-part + descriptor linking
    contextual = apply_contextual_rules(text, found)
    found.extend(contextual)
    return found

# Rebuild the lookup with the new normalization (phrases get re-normalized)
_PHRASE_LOOKUP = []
for canonical, phrases in SYNONYMS.items():
    for phrase in phrases:
        _PHRASE_LOOKUP.append((normalize(phrase), canonical))
_PHRASE_LOOKUP.sort(key=lambda x: -len(x[0]))

print("Extractor upgraded: contractions expanded, contextual rules active")

Built 128 contextual rules
Extractor upgraded: contractions expanded, contextual rules active


In [28]:
def apply_contextual_rules(text: str, already_found: list) -> list:
    """
    Within each sentence, if a body part appears AND a descriptor appears anywhere
    in that sentence, add the linked symptom. Body parts are tracked at sentence level.
    """
    already_set = set(already_found)
    new_finds = []
    # Split ONLY on real sentence boundaries, NOT on 'and'
    sentences = re.split(r"[.!?;]", text)
    for sent in sentences:
        sent_padded = f" {sent} "
        parts_in_sent = [p for p in BODY_PARTS if f" {p} " in sent_padded]
        if not parts_in_sent:
            continue
        for part in parts_in_sent:
            for (rule_part, descriptor), canonical in CONTEXTUAL_RULES.items():
                if rule_part != part:
                    continue
                if f" {descriptor} " in sent_padded and canonical not in already_set:
                    new_finds.append(canonical)
                    already_set.add(canonical)
    return new_finds

print("Contextual linker fixed: no longer splits on 'and'")

Contextual linker fixed: no longer splits on 'and'


In [29]:
tests = [
    "My elbow hurts and it's swollen",
    "My knee hurts and it's swollen and stiff",
    "my ankle is weak and swollen",
    "I have a terrible headache and I feel really dizzy",  # regression
]

for sent in tests:
    result = predict_from_text(sent, k=3)
    print(f"\nUser: {sent}")
    print(f"  Extracted: {result['symptoms']}")
    print(f"  Top 3:")
    for name, prob in result["predictions"]:
        print(f"    {name}: {prob:.3f}")


User: My elbow hurts and it's swollen
  Extracted: ['elbow pain', 'elbow swelling']
  Top 3:
    dislocation of the elbow: 0.508
    injury to the arm: 0.199
    bursitis: 0.115

User: My knee hurts and it's swollen and stiff
  Extracted: ['knee pain', 'knee swelling', 'knee stiffness or tightness']
  Top 3:
    injury to the leg: 0.521
    joint effusion: 0.154
    knee ligament or meniscus tear: 0.125

User: my ankle is weak and swollen
  Extracted: ['weakness', 'ankle swelling', 'ankle weakness']
  Top 3:
    fracture of the ankle: 0.818
    fracture of the leg: 0.015
    injury to the leg: 0.013

User: I have a terrible headache and I feel really dizzy
  Extracted: ['headache', 'dizziness']
  Top 3:
    autonomic nervous system disorder: 0.157
    atelectasis: 0.052
    cerebral edema: 0.037


In [30]:
import sys
sys.path.insert(0, "/scratch/budayaku/SML/src")
if "extractor" in sys.modules:
    del sys.modules["extractor"]
import extractor

for sentence in [
    "I have a headache and feel dizzy",
    "My elbow hurts and it's swollen",
    "burning when I pee and peeing a lot",
]:
    result = extractor.predict_from_text(sentence, k=3)
    print(f"\n> {sentence}")
    print(f"  symptoms: {result['symptoms']}")
    for name, p in result["predictions"]:
        print(f"  {name}: {p:.3f}")


> I have a headache and feel dizzy
  symptoms: ['headache', 'dizziness']
  autonomic nervous system disorder: 0.157
  atelectasis: 0.052
  cerebral edema: 0.037

> My elbow hurts and it's swollen
  symptoms: ['elbow pain', 'elbow swelling']
  dislocation of the elbow: 0.508
  injury to the arm: 0.199
  bursitis: 0.115

> burning when I pee and peeing a lot
  symptoms: ['painful urination', 'frequent urination']
  bladder disorder: 0.479
  temporary or benign blood in urine: 0.182
  benign vaginal discharge (leukorrhea): 0.064


In [2]:
!pip install --upgrade "transformers>=5.0.0rc0" torch accelerate --quiet

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
torchvision 0.25.0 requires torch==2.10.0, but you have torch 2.11.0 which is incompatible.


In [2]:
!pip uninstall torchvision -y --quiet

In [4]:
!pip install --upgrade torchvision --index-url https://download.pytorch.org/whl/cu130 --quiet

In [5]:
import sys
print("Python env:", sys.executable)

import torch, torchvision, transformers
print("torch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("transformers:", transformers.__version__)
print("CUDA:", torch.cuda.is_available())

from transformers import NanoChatConfig, NanoChatForCausalLM
print("✓ NanoChat classes import cleanly")

Python env: /home/budayaku/miniconda3/envs/yolo/bin/python
torch: 2.11.0+cu130
torchvision: 0.26.0+cu130
transformers: 5.5.4
CUDA: True
✓ NanoChat classes import cleanly


In [5]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "nanochat-students/d20-chat-transformers"

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (first run downloads ~1.1GB)...")
device = torch.device("cuda")
nanochat_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.bfloat16,
).to(device)
nanochat_model.eval()
print(f"✓ nanochat-d20 loaded on {device}")
print(f"Parameters: {sum(p.numel() for p in nanochat_model.parameters()) / 1e6:.1f}M")

/etc/python/sitecustomize.py:117: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  mod = _original_import(name, globals, locals, fromlist, level)


Loading tokenizer...


Loading model (first run downloads ~1.1GB)...


Loading weights: 100%|██████████| 122/122 [00:00<00:00, 9092.78it/s]


✓ nanochat-d20 loaded on cuda
Parameters: 561.0M


In [6]:
def chat(user_message: str, max_new_tokens: int = 150, temperature: float = 0.7) -> str:
    conversation = [{"role": "user", "content": user_message}]
    inputs = tokenizer.apply_chat_template(
        conversation,
        add_generation_prompt=True,
        tokenize=True,
        return_tensors="pt",
        return_dict=True,
    ).to(device)
    with torch.no_grad():
        outputs = nanochat_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.eos_token_id,
        )
    input_len = inputs["input_ids"].shape[1]
    reply = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
    return reply.strip()

print("--- Test 1: basic factual ---")
print("User: What is the capital of France?")
print("Bot:", chat("What is the capital of France?"))

print("\n--- Test 2: medical phrasing ---")
print("User: I have a headache and feel dizzy. What could it be?")
print("Bot:", chat("I have a headache and feel dizzy. What could it be?"))

print("\n--- Test 3: medical assistant style ---")
medical_prompt = """You are a friendly medical symptom assistant. A patient just said: "I have a headache and feel dizzy". Ask them ONE short follow-up question about other symptoms. Keep it under 25 words."""
print("Bot:", chat(medical_prompt))

--- Test 1: basic factual ---
User: What is the capital of France?
Bot: The capital of France is Paris.

--- Test 2: medical phrasing ---
User: I have a headache and feel dizzy. What could it be?
Bot: Headaches can be a common symptom of many conditions, but they can also be caused by various underlying conditions. It's essential to consider all possible causes before making any conclusions. 

It's possible that your headache is due to stress or anxiety. This is especially true if you're experiencing frequent or persistent episodes of dizziness or nausea, which can be a sign of a more serious underlying condition. If you're experiencing these symptoms frequently and for an extended period, it's crucial to consult with a healthcare professional for proper evaluation and treatment.

Another possibility is that your headache is related to a vitamin or mineral deficiency. This can be due to certain medications, certain health conditions, or a general lack of certain nutrients. If you're ex

In [7]:
# ---- Day 4: Conversation orchestrator ----
import sys
sys.path.insert(0, "/scratch/budayaku/SML/src")
if "extractor" in sys.modules:
    del sys.modules["extractor"]
import extractor

# Emergency keywords — if any appear, override everything with urgent advice
EMERGENCY_KEYWORDS = [
    "crushing chest pain", "chest pain and left arm",
    "can't move", "cannot move", "face drooping", "slurring my words",
    "suicidal", "kill myself", "want to die", "end my life",
    "severe bleeding", "unconscious", "not breathing",
    "overdose", "poisoning",
]

def check_emergency(text: str) -> bool:
    t = text.lower()
    return any(kw in t for kw in EMERGENCY_KEYWORDS)

EMERGENCY_REPLY = (
    "⚠️ What you're describing could be a medical emergency. "
    "Please call emergency services immediately — 911 in the US, 108 in India, "
    "or your local emergency number. Do not wait."
)

# Session state lives in a dict — easy to pass around
def new_session():
    return {
        "symptoms": [],         # accumulated canonical symptom names
        "turn": 0,              # how many exchanges so far
        "user_history": [],     # all user messages in this session
    }

MIN_SYMPTOMS_TO_PREDICT = 3
MIN_CONFIDENCE_TO_PREDICT = 0.20
MAX_TURNS_BEFORE_FORCE_PREDICT = 5


def generate_followup(current_symptoms: list) -> str:
    """Use nanochat to ask a follow-up question given current symptoms."""
    if current_symptoms:
        sym_str = ", ".join(current_symptoms)
        prompt = (
            f"You are a caring medical assistant helping gather symptoms. "
            f"The patient has mentioned: {sym_str}. "
            f"Ask them ONE short follow-up question to learn about any other symptoms "
            f"they might have. Be warm and brief. Do not repeat symptoms already mentioned."
        )
    else:
        prompt = (
            "You are a caring medical assistant. The patient hasn't described "
            "specific symptoms yet. Ask them ONE short question about what they're "
            "feeling. Be warm and brief."
        )
    return chat(prompt, max_new_tokens=60, temperature=0.7)


def generate_diagnosis_reply(symptoms: list, predictions: list) -> str:
    """Use nanochat to phrase the top-3 predictions naturally."""
    top3 = predictions[:3]
    sym_str = ", ".join(symptoms)
    pred_str = "; ".join(f"{name} ({prob*100:.0f}% match)" for name, prob in top3)
    prompt = (
        f"You are a caring medical assistant. Based on the symptoms ({sym_str}), "
        f"the most likely conditions are: {pred_str}. "
        f"Tell the patient this in a warm, clear way in 2-3 sentences. "
        f"End by strongly recommending they see a real doctor for diagnosis. "
        f"Do not add extra medical details you aren't told."
    )
    return chat(prompt, max_new_tokens=180, temperature=0.6)


def chatbot_turn(user_msg: str, state: dict) -> tuple:
    """One turn of conversation. Returns (bot_reply, updated_state)."""
    state["turn"] += 1
    state["user_history"].append(user_msg)

    # 1. Emergency check
    if check_emergency(user_msg):
        return EMERGENCY_REPLY, state

    # 2. Extract symptoms and merge into session state
    new_symptoms = extractor.extract_symptoms(user_msg)
    for s in new_symptoms:
        if s not in state["symptoms"]:
            state["symptoms"].append(s)

    # 3. Decide: predict or ask?
    should_predict = False
    predictions = []
    if len(state["symptoms"]) >= MIN_SYMPTOMS_TO_PREDICT:
        predictions = extractor.predict_from_symptom_list(state["symptoms"], k=5)
        if predictions and predictions[0][1] >= MIN_CONFIDENCE_TO_PREDICT:
            should_predict = True
    if state["turn"] >= MAX_TURNS_BEFORE_FORCE_PREDICT and state["symptoms"]:
        # Force a prediction to avoid endless follow-ups
        predictions = extractor.predict_from_symptom_list(state["symptoms"], k=5)
        should_predict = True

    if should_predict:
        reply = generate_diagnosis_reply(state["symptoms"], predictions)
        reply += f"\n\n(Symptoms I noted: {', '.join(state['symptoms'])})"
    else:
        reply = generate_followup(state["symptoms"])

    return reply, state


# ---- Smoke test: simulate a 4-turn conversation ----
session = new_session()
conversation = [
    "I've had a terrible headache for two days",
    "Yes, and I feel really dizzy sometimes",
    "I've also been throwing up this morning",
    "My neck feels stiff too",
]

for user_msg in conversation:
    print(f"\nUser: {user_msg}")
    reply, session = chatbot_turn(user_msg, session)
    print(f"Bot:  {reply}")
    print(f"[symptoms so far: {session['symptoms']}]")


User: I've had a terrible headache for two days
Bot:  *Sigh* I'm glad to hear you're feeling a bit dizzy and lightheaded. That's a common symptom of a variety of conditions, and it's not something I'd like to repeat. I'm here to listen and help you get to the bottom of it. Can you tell me more
[symptoms so far: ['headache']]

User: Yes, and I feel really dizzy sometimes
Bot:  "Okay, here are some additional symptoms that could be related to your headache. Can you tell me what you think might be causing it?"
[symptoms so far: ['headache', 'dizziness']]

User: I've also been throwing up this morning
Bot:  "Okay, thank you for coming in. I'm glad you're feeling better. Can you tell me what you've been experiencing lately? Was it dizziness, nausea, or vomiting?"
[symptoms so far: ['headache', 'dizziness', 'vomiting']]

User: My neck feels stiff too
Bot:  A tension headache is a common symptom of meningitis, a condition that can cause severe head pain and sensitivity to light. If you're ex

In [8]:
# Diagnostic: test what XGBoost predicts for the final symptom set
test_symptoms = ['headache', 'dizziness', 'vomiting', 'neck stiffness or tightness']
preds = extractor.predict_from_symptom_list(test_symptoms, k=5)
print("XGBoost top 5 predictions for meningitis-triad symptoms:")
for name, prob in preds:
    print(f"  {name}: {prob:.3f}")

XGBoost top 5 predictions for meningitis-triad symptoms:
  tension headache: 0.368
  pain disorder affecting the neck: 0.267
  meningitis: 0.247
  torticollis: 0.017
  heat exhaustion: 0.009


In [9]:
# Is the basic chatbot_turn function still working?
test_session = new_session()
test_reply, test_session = chatbot_turn("I have a headache and feel dizzy", test_session)
print("Reply:", test_reply[:200])
print("Symptoms:", test_session["symptoms"])

Reply: Yes, I'll keep an eye on you and ask you to keep an eye on your head. Any dizziness or headache, please. Can you tell me when they first started?
Symptoms: ['headache', 'dizziness']


In [10]:
session = new_session()

print("=" * 70)
print("🩺  MEDICAL SYMPTOM ASSISTANT")
print("=" * 70)
print("Type your symptoms. Type 'quit' when done.")
print("=" * 70)

while True:
    user_msg = input("👤 You: ").strip()
    if not user_msg:
        continue
    if user_msg.lower() in ("quit", "exit", "stop"):
        print("Goodbye! 🩺")
        break
    if user_msg.lower() == "reset":
        session = new_session()
        print("[🔄 New conversation]")
        continue
    reply, session = chatbot_turn(user_msg, session)
    print(f"\n🤖 Bot: {reply}")
    print(f"   [symptoms: {session['symptoms']}]\n")

🩺  MEDICAL SYMPTOM ASSISTANT
Type your symptoms. Type 'quit' when done.

🤖 Bot: Shortness of breath, dizziness, and nausea are all concerning symptoms we're already looking into. Can you please confirm if these symptoms are present and if they're severe enough to be causing concern?
   [symptoms: ['shortness of breath', 'dizziness']]


🤖 Bot: Yes, shortness of breath, dizziness, or both.
   [symptoms: ['shortness of breath', 'dizziness']]


🤖 Bot: Yes
   [symptoms: ['shortness of breath', 'dizziness']]


🤖 Bot: Yes, please provide the next follow-up question, and I will respond with more information.
   [symptoms: ['shortness of breath', 'dizziness']]


🤖 Bot: I'm here to help you manage your symptoms. Pneumoconiosis is a condition that can cause shortness of breath, dizziness, and other symptoms. Paroxysmal ventricular tachycardia is a heart rhythm disorder that can cause brief, irregular heartbeats. Paroxysmal supraventricular tachycardia is a type of heart rhythm disorder that can c